# Building Regressors

This notebook demonstrates how to build regressors for fMRI analysis using pyfmridesign.

Author: Bradley R. Buchsbaum (Python port)  
Date: 2024

## Introduction: What is a Regressor?

In fMRI analysis, a **regressor** (or predictor) represents the expected BOLD signal timecourse associated with a specific experimental condition or event type. It's typically created by convolving a series of event onsets (often represented as delta functions or "sticks") with a hemodynamic response function (HRF).

`pyfmridesign` provides the `regressor()` function to easily create these objects from event timings and an HRF. While these regressor objects are often constructed automatically behind the scenes by modeling functions, this notebook explores how to create and manipulate them directly, offering finer control over the model components.

In [ ]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pyfmridesign import (
    HRF_SPMG1, regressor, SamplingFrame, samples,
    gen_hrf, global_onsets
)

# Set up plotting style
plt.style.use('seaborn-v0_8-darkgrid')

## Basic Regressor from Event Onsets

Suppose we have a simple event-related fMRI design with stimuli presented every 12 seconds. We want to model these events using the SPM canonical HRF. The events are brief, so we model them with a duration of 0 seconds (instantaneous).

In [ ]:
# Define event onsets (11 events, every 12 seconds)
onsets = np.arange(0, 11 * 12, 12)
print(f"Event onsets: {onsets}")

# Create the regressor object
# Uses HRF_SPMG1 by default if no hrf is specified
# Duration is 0 by default
reg1 = regressor(onsets=onsets, hrf=HRF_SPMG1)

# The regressor function returns the convolved signal
print(f"Regressor shape: {reg1.shape}")
print(f"Regressor type: {type(reg1)}")

## Specifying the Sampling Frame

To evaluate the regressor at specific time points (e.g., at each fMRI scan), we need to provide a sampling frame that defines the temporal structure of our data:

In [ ]:
# Create a sampling frame: 100 scans, TR = 2 seconds
sframe = SamplingFrame(n_scans=100, tr=2.0)

# Now create the regressor with explicit sampling
reg_sampled = regressor(
    onsets=onsets,
    hrf=HRF_SPMG1,
    sampling_frame=sframe
)

print(f"Sampled regressor length: {len(reg_sampled)} (matches n_scans)")
print(f"Total duration: {100 * 2} seconds")

# Get the time points for plotting
time_points = samples(sframe)

# Visualize the regressor
plt.figure(figsize=(12, 6))
plt.plot(time_points, reg_sampled, 'b-', linewidth=2)

# Mark event onsets
for onset in onsets:
    if onset < time_points[-1]:
        plt.axvline(x=onset, color='red', linestyle='--', alpha=0.5)

plt.xlabel('Time (seconds)')
plt.ylabel('Signal Amplitude')
plt.title('Regressor for Periodic Events (TR = 2s)')
plt.grid(True, alpha=0.3)
plt.show()

## Events with Variable Durations

Real experiments often have events with different durations. The `regressor()` function can handle this by modeling events as boxcars convolved with the HRF:

In [ ]:
# Events with variable durations
event_data = {
    'onsets': [10, 30, 50, 70, 90, 110, 130],
    'durations': [2, 5, 1, 3, 4, 2, 6]  # Duration in seconds
}

# Create sampling frame for longer experiment
sframe_long = SamplingFrame(n_scans=100, tr=2.0)

# Create regressor with durations
reg_duration = regressor(
    onsets=event_data['onsets'],
    durations=event_data['durations'],
    hrf=HRF_SPMG1,
    sampling_frame=sframe_long
)

# For comparison, create regressor without durations (instantaneous events)
reg_instant = regressor(
    onsets=event_data['onsets'],
    hrf=HRF_SPMG1,
    sampling_frame=sframe_long
)

# Plot comparison
time_points = samples(sframe_long)

plt.figure(figsize=(14, 8))

# Plot with durations
plt.subplot(2, 1, 1)
plt.plot(time_points, reg_duration, 'b-', linewidth=2, label='With durations')

# Show event blocks
for onset, duration in zip(event_data['onsets'], event_data['durations']):
    plt.axvspan(onset, onset + duration, alpha=0.3, color='red')

plt.ylabel('Signal Amplitude')
plt.title('Regressor with Event Durations')
plt.legend()
plt.grid(True, alpha=0.3)

# Plot without durations
plt.subplot(2, 1, 2)
plt.plot(time_points, reg_instant, 'g-', linewidth=2, label='Instantaneous')

# Show event onsets
for onset in event_data['onsets']:
    plt.axvline(x=onset, color='red', linestyle='--', alpha=0.5)

plt.xlabel('Time (seconds)')
plt.ylabel('Signal Amplitude')
plt.title('Regressor with Instantaneous Events')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Multi-run Experiments with Block Structure

fMRI experiments often consist of multiple runs or blocks. The `global_onsets()` function helps manage events across runs:

In [ ]:
# Define events for a 3-run experiment
# Each run has different event timings
run1_onsets = [5, 15, 25, 35]
run2_onsets = [8, 18, 28, 38]
run3_onsets = [10, 20, 30, 40]

# Create a multi-block sampling frame
# 3 runs, 50 scans each, TR = 2s
sframe_multi = SamplingFrame(blocklens=[50, 50, 50], tr=2.0)

# Convert local onsets to global onsets
global_onsets_all = global_onsets(
    [run1_onsets, run2_onsets, run3_onsets],
    sampling_frame=sframe_multi
)

print(f"Global onsets: {global_onsets_all}")

# Create regressor for multi-run data
reg_multirun = regressor(
    onsets=global_onsets_all,
    hrf=HRF_SPMG1,
    sampling_frame=sframe_multi
)

# Plot the multi-run regressor
time_points = samples(sframe_multi)

plt.figure(figsize=(14, 6))
plt.plot(time_points, reg_multirun, 'b-', linewidth=2)

# Mark run boundaries
run_boundaries = np.cumsum([0] + [50, 50]) * 2  # Convert to seconds
for boundary in run_boundaries[1:-1]:
    plt.axvline(x=boundary, color='black', linestyle='-', linewidth=2, alpha=0.5)

# Mark events
for onset in global_onsets_all:
    plt.axvline(x=onset, color='red', linestyle='--', alpha=0.5)

plt.xlabel('Time (seconds)')
plt.ylabel('Signal Amplitude')
plt.title('Multi-run Regressor (3 runs, 50 scans each)')
plt.grid(True, alpha=0.3)

# Add run labels
for i, (start, end) in enumerate(zip(run_boundaries[:-1], run_boundaries[1:])):
    plt.text((start + end) / 2, plt.ylim()[1] * 0.9, f'Run {i+1}', 
             ha='center', va='center', fontsize=12, fontweight='bold')

plt.show()

## Parametric Modulation

Sometimes we want to model how the BOLD response varies with a continuous parameter (e.g., stimulus intensity, reaction time). This is done by weighting each event:

In [ ]:
# Event onsets and parametric weights
onsets = [10, 25, 40, 55, 70, 85, 100, 115, 130, 145]
# Simulate reaction times (in seconds) - faster RTs might indicate easier trials
reaction_times = [1.2, 0.8, 1.5, 0.7, 1.1, 0.9, 1.4, 0.6, 1.3, 1.0]

# Normalize weights (mean-centered)
weights = np.array(reaction_times)
weights = weights - weights.mean()

# Create sampling frame
sframe = SamplingFrame(n_scans=100, tr=2.0)

# Create weighted regressor
reg_weighted = regressor(
    onsets=onsets,
    weights=weights,
    hrf=HRF_SPMG1,
    sampling_frame=sframe
)

# Create unweighted regressor for comparison
reg_unweighted = regressor(
    onsets=onsets,
    hrf=HRF_SPMG1,
    sampling_frame=sframe
)

# Plot comparison
time_points = samples(sframe)

plt.figure(figsize=(14, 10))

# Plot weighted regressor
plt.subplot(3, 1, 1)
plt.plot(time_points, reg_weighted, 'b-', linewidth=2)
plt.ylabel('Signal Amplitude')
plt.title('Parametrically Modulated Regressor (by Reaction Time)')
plt.grid(True, alpha=0.3)

# Add weight annotations
for onset, weight, rt in zip(onsets, weights, reaction_times):
    if onset < time_points[-1]:
        plt.axvline(x=onset, color='red', linestyle='--', alpha=0.3)
        plt.text(onset, plt.ylim()[1] * 0.8, f'{rt:.1f}s', 
                 rotation=90, ha='right', va='bottom', fontsize=8)

# Plot unweighted regressor
plt.subplot(3, 1, 2)
plt.plot(time_points, reg_unweighted, 'g-', linewidth=2)
plt.ylabel('Signal Amplitude')
plt.title('Unweighted Regressor')
plt.grid(True, alpha=0.3)

for onset in onsets:
    if onset < time_points[-1]:
        plt.axvline(x=onset, color='red', linestyle='--', alpha=0.3)

# Plot difference
plt.subplot(3, 1, 3)
plt.plot(time_points, reg_weighted - reg_unweighted, 'purple', linewidth=2)
plt.ylabel('Difference')
plt.xlabel('Time (seconds)')
plt.title('Difference (Weighted - Unweighted)')
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='black', linestyle='-', alpha=0.5)

plt.tight_layout()
plt.show()

## Using Different HRF Models

The choice of HRF can significantly impact the regressor. Let's compare regressors created with different HRF models:

In [ ]:
# Define a set of events
onsets = [10, 30, 50, 70, 90, 110, 130]

# Create different HRFs
hrfs = {
    'SPM Canonical': HRF_SPMG1,
    'SPM + Derivative': gen_hrf(type="spmg1", include_deriv=1),
    'FIR (5 bins)': gen_hrf(type="fir", nbasis=5, span=10),
    'Gaussian': gen_hrf(type="gaussian", width=4, center=5)
}

# Create sampling frame
sframe = SamplingFrame(n_scans=100, tr=2.0)
time_points = samples(sframe)

# Create and plot regressors
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for idx, (name, hrf) in enumerate(hrfs.items()):
    # Create regressor
    reg = regressor(
        onsets=onsets,
        hrf=hrf,
        sampling_frame=sframe
    )
    
    # Handle multi-column output (e.g., derivatives, FIR)
    if reg.ndim > 1:
        # Plot each basis function
        for i in range(reg.shape[1]):
            label = f'Basis {i+1}' if i > 0 else 'Main'
            axes[idx].plot(time_points, reg[:, i], linewidth=2, label=label)
        axes[idx].legend()
    else:
        axes[idx].plot(time_points, reg, 'b-', linewidth=2)
    
    # Mark events
    for onset in onsets:
        if onset < time_points[-1]:
            axes[idx].axvline(x=onset, color='red', linestyle='--', alpha=0.3)
    
    axes[idx].set_title(f'{name} HRF')
    axes[idx].set_xlabel('Time (seconds)')
    axes[idx].set_ylabel('Signal Amplitude')
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Regressors with Different HRF Models', fontsize=16)
plt.tight_layout()
plt.show()

## Summary

In this notebook, we've covered:

1. **Basic regressors** - Creating regressors from event onsets
2. **Sampling frames** - Specifying the temporal structure of fMRI data
3. **Event durations** - Modeling extended events vs. instantaneous events
4. **Multi-run experiments** - Handling events across multiple runs/blocks
5. **Parametric modulation** - Weighting events by continuous parameters
6. **HRF models** - How different HRFs affect the resulting regressors

Key takeaways:
- Regressors are the building blocks of fMRI design matrices
- The choice of HRF and modeling decisions (durations, weights) significantly impact the analysis
- `pyfmridesign` provides flexible tools for creating custom regressors
- Multiple basis functions (derivatives, FIR) allow for more flexible modeling of the hemodynamic response